# Lakehouse Shortcut Audit: Multipart Delta Checkpoints

This notebook checks whether the **current Fabric workspace** has lakehouse table shortcuts that point to Delta tables with multipart checkpoints (`_last_checkpoint.parts > 1`).

If multipart usage is found, the notebook prints the matching shortcuts.

In [ ]:
import json
from concurrent.futures import ThreadPoolExecutor, as_completed

from sempy.fabric import FabricRestClient

try:
    import notebookutils
    runtime_context = notebookutils.runtime.context
    fs = notebookutils.fs

    def get_current_workspace_id():
        return runtime_context.get("currentWorkspaceId")
except Exception:
    import mssparkutils
    runtime_context = mssparkutils.runtime.context()
    fs = mssparkutils.fs

    def get_current_workspace_id():
        try:
            return runtime_context["currentWorkspaceId"]
        except Exception:
            return None

In [ ]:
client = FabricRestClient()
WORKSPACE_ID = get_current_workspace_id()
MAX_WORKERS = 16

if not WORKSPACE_ID:
    raise ValueError("Workspace ID not found in runtime context. Run inside Fabric workspace notebook.")

def get_paged(path: str):
    return list(client.get_paged(path))

def list_lakehouses(workspace_id: str):
    items = get_paged(f"/v1/workspaces/{workspace_id}/items")
    return [item for item in items if str(item.get("type", "")).lower().startswith("lakehouse")]

def list_shortcuts(workspace_id: str, item_id: str):
    return get_paged(f"/v1/workspaces/{workspace_id}/items/{item_id}/shortcuts")

def read_checkpoint_parts(target_workspace_id: str, target_item_id: str, target_tables_path: str):
    table_rel_path = target_tables_path.strip("/")
    checkpoint_uri = (
        f"abfss://{target_workspace_id}@onelake.dfs.fabric.microsoft.com/"
        f"{target_item_id}/{table_rel_path}/_delta_log/_last_checkpoint"
    )
    try:
        raw = fs.head(checkpoint_uri, 4096)
        checkpoint = json.loads(raw)
        return int(checkpoint.get("parts", 0))
    except Exception:
        return None

In [ ]:
shortcut_refs = []  # (lakehouse_name, shortcut_path, target_key)
targets = set()     # (workspace_id, item_id, table_path)

for lakehouse in list_lakehouses(WORKSPACE_ID):
    lakehouse_id = lakehouse.get("id")
    lakehouse_name = lakehouse.get("displayName", "")
    if not lakehouse_id:
        continue

    try:
        shortcuts = list_shortcuts(WORKSPACE_ID, lakehouse_id)
    except Exception:
        continue

    for shortcut in shortcuts:
        shortcut_path = shortcut.get("path", "") or ""
        if not shortcut_path.startswith("Tables/"):
            continue

        target = shortcut.get("target") or {}
        if (target.get("type") or "") != "OneLake":
            continue

        one_lake = target.get("oneLake") or {}
        target_workspace_id = one_lake.get("workspaceId")
        target_item_id = one_lake.get("itemId")
        target_path = one_lake.get("path") or ""

        if not (target_workspace_id and target_item_id and target_path.startswith("Tables/")):
            continue

        key = (target_workspace_id, target_item_id, target_path)
        targets.add(key)
        shortcut_refs.append((lakehouse_name, shortcut_path, key))

target_parts = {}

def check_one(target_key):
    ws_id, item_id, table_path = target_key
    return target_key, read_checkpoint_parts(ws_id, item_id, table_path)

if targets:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = [executor.submit(check_one, key) for key in targets]
        for future in as_completed(futures):
            key, parts = future.result()
            target_parts[key] = parts

In [ ]:
multipart_shortcuts = []

for lakehouse_name, shortcut_path, key in shortcut_refs:
    parts = target_parts.get(key)
    if parts is not None and parts > 1:
        multipart_shortcuts.append((lakehouse_name, shortcut_path))

print(f"Workspace: {WORKSPACE_ID}")
print(f"Scanned table shortcuts in lakehouses: {len(shortcut_refs)}")
print(f"Unique target tables checked: {len(targets)}")
print(f"Shortcuts pointing to multipart checkpoints (parts > 1): {len(multipart_shortcuts)}")

if multipart_shortcuts:
    print("\nMultipart checkpoint usage detected (Lakehouse | ShortcutPath):")
    for lakehouse_name, shortcut_path in multipart_shortcuts:
        print(f"{lakehouse_name} | {shortcut_path}")
else:
    print("No multipart checkpoint usage detected in lakehouse table shortcuts.")